# **STEP 1: Import Required Libraries**

In [12]:
import pandas as pd
import re
import string
import nltk

# Download required NLTK data
print("Downloading NLTK resources...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

print("✓ All libraries imported successfully!\n")

✓ All libraries imported successfully!



# **STEP 2: Create Sample Data with Messy Text**

In [13]:
# Sample messy text data
data = {
    'id': [1, 2, 3, 4, 5],
    'text': [
        "Hey!!!  Check out this website: https://example.com 😊 It's AMAZING!!!",
        "Natural Language Processing is SO cool! 🚀🚀 Visit www.nlp.org for more info   ",
        "I'm learning Python!!!   It's really    helpful for DATA analysis 📊",
        "Machine Learning models are    POWERFUL!! See: http://ml-info.com 💡",
        "Text preprocessing removes    noise!!!  😃 It makes data CLEAN and usable..."
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save original data
df.to_csv('messy_text_data.csv', index=False)
print("✓ Sample dataset created and saved as 'messy_text_data.csv'")
print("\nOriginal Data:")
print(df)

✓ Sample dataset created and saved as 'messy_text_data.csv'

Original Data:
   id                                               text
0   1  Hey!!!  Check out this website: https://exampl...
1   2  Natural Language Processing is SO cool! 🚀🚀 Vis...
2   3  I'm learning Python!!!   It's really    helpfu...
3   4  Machine Learning models are    POWERFUL!! See:...
4   5  Text preprocessing removes    noise!!!  😃 It m...


# **STEP 3: Define Preprocessing Functions**

In [14]:
def show_example(step_name, original, processed, row=0):
    """Helper function to display before/after for a specific row"""
    print(f"\n{step_name}")
    print("-" * 70)
    print(f"Before: {original[row]}")
    print(f"After:  {processed[row]}")

# **STEP 4: Preprocessing Pipeline**

In [15]:
# Make a copy for preprocessing
df_clean = df.copy()

**4.1: Lowercasing**

In [16]:
original = df_clean['text'].copy()
df_clean['text'] = df_clean['text'].str.lower()
show_example("Lowercasing Example:", original, df_clean['text'])


Lowercasing Example:
----------------------------------------------------------------------
Before: Hey!!!  Check out this website: https://example.com 😊 It's AMAZING!!!
After:  hey!!!  check out this website: https://example.com 😊 it's amazing!!!


**4.2: Remove URLs**

In [17]:
original = df_clean['text'].copy()

url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
www_pattern = r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

df_clean['text'] = df_clean['text'].str.replace(url_pattern, '', regex=True)
df_clean['text'] = df_clean['text'].str.replace(www_pattern, '', regex=True)

show_example("URL Removal Example:", original, df_clean['text'], row=1)


URL Removal Example:
----------------------------------------------------------------------
Before: natural language processing is so cool! 🚀🚀 visit www.nlp.org for more info   
After:  natural language processing is so cool! 🚀🚀 visit  for more info   


**4.3: Remove Emojis**

In [18]:
original = df_clean['text'].copy()

emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

df_clean['text'] = df_clean['text'].apply(lambda x: emoji_pattern.sub(r'', x))

show_example("Emoji Removal Example:", original, df_clean['text'], row=0)


Emoji Removal Example:
----------------------------------------------------------------------
Before: hey!!!  check out this website:  😊 it's amazing!!!
After:  hey!!!  check out this website:   it's amazing!!!


**4.4: Remove Punctuation**

In [19]:
original = df_clean['text'].copy()

df_clean['text'] = df_clean['text'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

show_example("Punctuation Removal Example:", original, df_clean['text'], row=0)


Punctuation Removal Example:
----------------------------------------------------------------------
Before: hey!!!  check out this website:   it's amazing!!!
After:  hey  check out this website   its amazing


**4.5: Remove Extra Spaces**

In [20]:
original = df_clean['text'].copy()

df_clean['text'] = df_clean['text'].apply(lambda x: ' '.join(x.split()))

show_example("Extra Space Removal Example:", original, df_clean['text'], row=2)



Extra Space Removal Example:
----------------------------------------------------------------------
Before: im learning python   its really    helpful for data analysis 
After:  im learning python its really helpful for data analysis


**4.6: Tokenization**

In [21]:
df_clean['tokens'] = df_clean['text'].apply(word_tokenize)

print("Tokenization Example:")
print("-" * 70)

print(f"Text:   {df_clean['text'][0]}")
print(f"Tokens: {df_clean['tokens'][0]}")

Tokenization Example:
----------------------------------------------------------------------
Text:   hey check out this website its amazing
Tokens: ['hey', 'check', 'out', 'this', 'website', 'its', 'amazing']


**4.7: Remove Stopwords**

In [22]:
stop_words = set(stopwords.words('english'))
print(f"Number of stopwords: {len(stop_words)}")
print(f"Sample stopwords: {list(stop_words)[:10]}")

original_tokens = df_clean['tokens'].copy()
df_clean['tokens_no_stop'] = df_clean['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)
print("\nStopword Removal Example:")
print("-" * 70)
print(f"Before: {original_tokens[0]}")
print(f"After:  {df_clean['tokens_no_stop'][0]}")

Number of stopwords: 198
Sample stopwords: ['but', 'now', 'was', "he'll", "aren't", 'against', 'hers', 'this', 'herself', 'such']

Stopword Removal Example:
----------------------------------------------------------------------
Before: ['hey', 'check', 'out', 'this', 'website', 'its', 'amazing']
After:  ['hey', 'check', 'website', 'amazing']


**4.8: Lemmatization**

In [23]:
lemmatizer = WordNetLemmatizer()
df_clean['lemmatized'] = df_clean['tokens_no_stop'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)
print("Lemmatization Example:")
print("-" * 70)
print(f"Before: {df_clean['tokens_no_stop'][3]}")
print(f"After:  {df_clean['lemmatized'][3]}")

Lemmatization Example:
----------------------------------------------------------------------
Before: ['machine', 'learning', 'models', 'powerful', 'see']
After:  ['machine', 'learning', 'model', 'powerful', 'see']


**4.9: Stemming**

In [24]:
stemmer = PorterStemmer()
df_clean['stemmed'] = df_clean['tokens_no_stop'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)
print("Stemming Example:")
print("-" * 70)
print(f"Before: {df_clean['tokens_no_stop'][1]}")
print(f"After:  {df_clean['stemmed'][1]}")

Stemming Example:
----------------------------------------------------------------------
Before: ['natural', 'language', 'processing', 'cool', 'visit', 'info']
After:  ['natur', 'languag', 'process', 'cool', 'visit', 'info']


# **STEP 5: Prepare Final Output**

In [25]:
# Create final clean text from lemmatized tokens
df_clean['clean_text_lemmatized'] = df_clean['lemmatized'].apply(lambda x: ' '.join(x))
df_clean['clean_text_stemmed'] = df_clean['stemmed'].apply(lambda x: ' '.join(x))

# Prepare output DataFrame
output_df = pd.DataFrame({
    'id': df_clean['id'],
    'original_text': df['text'],
    'clean_text_lemmatized': df_clean['clean_text_lemmatized'],
    'clean_text_stemmed': df_clean['clean_text_stemmed']
})

print("\nFinal Comparison:")
print("-" * 70)
for idx in range(len(output_df)):
    print(f"\nText {idx + 1}:")
    print(f"  Original:    {output_df['original_text'][idx][:60]}...")
    print(f"  Lemmatized:  {output_df['clean_text_lemmatized'][idx]}")
    print(f"  Stemmed:     {output_df['clean_text_stemmed'][idx]}")



Final Comparison:
----------------------------------------------------------------------

Text 1:
  Original:    Hey!!!  Check out this website: https://example.com 😊 It's A...
  Lemmatized:  hey check website amazing
  Stemmed:     hey check websit amaz

Text 2:
  Original:    Natural Language Processing is SO cool! 🚀🚀 Visit www.nlp.org...
  Lemmatized:  natural language processing cool visit info
  Stemmed:     natur languag process cool visit info

Text 3:
  Original:    I'm learning Python!!!   It's really    helpful for DATA ana...
  Lemmatized:  im learning python really helpful data analysis
  Stemmed:     im learn python realli help data analysi

Text 4:
  Original:    Machine Learning models are    POWERFUL!! See: http://ml-inf...
  Lemmatized:  machine learning model powerful see
  Stemmed:     machin learn model power see

Text 5:
  Original:    Text preprocessing removes    noise!!!  😃 It makes data CLEA...
  Lemmatized:  text preprocessing remove noise make data clean usa

# **STEP 6: Save Results**

In [27]:
output_df.to_csv('clean_text_data.csv', index=False)
print("✓ Clean data saved as 'clean_text_data.csv'")

# Save detailed version with all intermediate steps
detailed_output = pd.DataFrame({
    'id': df_clean['id'],
    'original': df['text'],
    'lowercased': df_clean['text'],
    'tokens': df_clean['tokens'].apply(lambda x: ' '.join(x)),
    'no_stopwords': df_clean['tokens_no_stop'].apply(lambda x: ' '.join(x)),
    'lemmatized': df_clean['clean_text_lemmatized'],
    'stemmed': df_clean['clean_text_stemmed']
})
detailed_output.to_csv('detailed_preprocessing_steps.csv', index=False)
print("✓ Detailed steps saved as 'detailed_preprocessing_steps.csv'")

print("\n" + "="*70)
print("✓ TEXT PREPROCESSING COMPLETE!")
print("="*70)
print("\nOutput files created:")
print("  1. messy_text_data.csv - Original messy data")
print("  2. clean_text_data.csv - Final cleaned data")
print("  3. detailed_preprocessing_steps.csv - All intermediate steps")

✓ Clean data saved as 'clean_text_data.csv'
✓ Detailed steps saved as 'detailed_preprocessing_steps.csv'

✓ TEXT PREPROCESSING COMPLETE!

Output files created:
  1. messy_text_data.csv - Original messy data
  2. clean_text_data.csv - Final cleaned data
  3. detailed_preprocessing_steps.csv - All intermediate steps
